In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# repo root: works whether the notebook is opened from notebooks/ or from the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))     # lets us import the sandwich package even without pip install

from sandwich.config import load_problem
from sandwich.optimisation.problem import SandwichProblem
from sandwich.optimisation.run import load_moga_config, load_surrogate

cfg = load_moga_config()            # algorithm settings from config/moga.yaml
surrogate = load_surrogate(cfg)     # trained ANN + Eq. 16 scalers, loaded once
problem = SandwichProblem(surrogate)    # design space and limits, read from design_space.yaml once

import json
default = json.load(open(ROOT / cfg["output_dir"] / "metrics.json"))     # saved by 07a
print(f"07a default run: {default['cardinality']} points, {default['run_seconds']} s for {default['n_evaluations']} evaluations")

In [ ]:
# one entry per setting: a label and the moga.yaml values it changes
variants = {
    "centre": {},
    "pop_50":  {"algorithm": {"pop_size": 50}},
    "pop_200": {"algorithm": {"pop_size": 200}},
    "cx_0.6":  {"algorithm": {"crossover": {"prob": 0.6}}},
    "cx_0.8":  {"algorithm": {"crossover": {"prob": 0.8}}},
    "mut_1/16": {"algorithm": {"mutation": {"prob_var": 1 / 16}}},
    "mut_1/4":  {"algorithm": {"mutation": {"prob_var": 1 / 4}}},
    "eta_5":   {"algorithm": {"crossover": {"eta": 5}}},
    "eta_30":  {"algorithm": {"crossover": {"eta": 30}}},
    "select_random": {"algorithm": {"selection": "random"}},
    "budget_11500": {"algorithm": {"n_evaluations": 11500}},
    "budget_46000": {"algorithm": {"n_evaluations": 46000}},
}
SEEDS = [1, 2, 3, 4, 5]

# readable matrix: the full setting each variant runs with
from sandwich.optimisation.run import merge_config
rows = []
for label, ov in variants.items():
    a = merge_config(cfg, ov)["algorithm"]
    rows.append({"variant": label, "pop_size": a["pop_size"], "n_evaluations": a["n_evaluations"],
                 "generations_approx": a["n_evaluations"] // a["pop_size"],
                 "crossover_prob": a["crossover"]["prob"], "sbx_eta": a["crossover"]["eta"],
                 "mutation_prob_var": a["mutation"]["prob_var"] or 1 / 8, "selection": a["selection"]})
matrix = pd.DataFrame(rows).set_index("variant")

n_runs = len(variants) * len(SEEDS)
est = n_runs * default["run_seconds"] * (matrix.n_evaluations.mean() / default["n_evaluations"])
print(f"{len(variants)} settings x {len(SEEDS)} seeds = {n_runs} runs, roughly {est / 60:.1f} min")

out = ROOT / "results/moga"
matrix.to_csv(out / "sweep_matrix.csv")
with open(out / "sweep_variants.json", "w") as f:
    json.dump({"variants": variants, "seeds": SEEDS}, f, indent=2)
matrix

In [ ]:
from sandwich.optimisation.run import run_sweep

smoke = run_sweep({"centre": {}, "pop_50": variants["pop_50"]}, seeds=[1],
                  base_cfg=merge_config(cfg, {"algorithm": {"n_evaluations": 3000}}), surrogate=surrogate)
smoke[["variant", "seed", "n_evaluations", "cardinality", "distinct_designs", "spread_delta", "violation_rate", "knee_W_t", "knee_C_t"]]